# 2. Link tickets to the bus network
Tickets use stop **names**; GTFS uses stop **ids**. In the real data the route name already carries the direction (`8UP`, `8ADOWN` are separate GTFS routes), so there is no direction guessing.
1. Build each route's ordered stop list from GTFS (a *pattern*).
2. Fuzzy-match ticket stop names to that route's stops.
3. Find each ticket's boarding and alighting position along the route.

In [ ]:
import sys, os
sys.path.append("..")                       # lets the notebook import config.py and the crowd/ folder
# os.environ["DATA_MODE"] = "synthetic"     # <- uncomment to try everything on the demo data
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import config
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
P = config.PROC; print("mode:", config.MODE, "| processed folder:", P)

In [ ]:
from crowd import gtfs, matching, trips
tk = pd.read_parquet(P / 'tickets.parquet')
print('share of tickets WITHOUT a vehicle number:', round(tk.vehicle.isna().mean(), 3), '-> we cannot rebuild trips from vehicles, so the model uses flow instead (notebook 3)')
pat, hourly, stops = gtfs.build_patterns()      # reads data/raw/gtfs/*.txt
print(pat.groupby(['route_id', 'dir']).ngroups, 'route-directions;', pat.stop_id.nunique(), 'stops')
pat.head(8)

In [ ]:
sizes = pat.groupby(['route', 'dir']).idx.max().add(1)
sns.histplot(sizes, bins=30); plt.xlabel('stops per route'); plt.show()

## Matching stop names
GTFS names are often composite, e.g. `Laxmi Nagar Metro Station / Laxmi Nagar`, while tickets say just `Laxmi Nagar`. So every `/` part becomes an alias. Matching goes in tiers: exact alias, fuzzy (RapidFuzz), contained-in-name, ignoring spaces, and finally any stop in Delhi.

In [ ]:
from crowd.text import norm_name
for s in ['S.J. Hospital', 'Laxmi Nagar X-ing', 'E.S.I. Hospital']: print(repr(s), '->', repr(norm_name(s)))
sm = matching.match_stop_names(tk, pat)
print(matching.match_report(sm))

In [ ]:
share = sm.groupby('method').n_uses.sum().rename('uses').reset_index()
sns.barplot(data=share, y='method', x='uses'); plt.title('How stop names were matched'); plt.show()
sm[sm.gtfs_name_n.isna() & (sm.method != 'blank')].sort_values('n_uses', ascending=False).head(20)   # fix these in crowd/text.py or matching.py

In [ ]:
tx, stats = trips.resolve_tickets(tk, pat, sm)
print(stats)
tx[['route_n', 'dir', 'b', 'a', 'pax', 'ts', 'imputed_end']].head(10)   # b = boarding position, a = alighting position along the route

`imputed_end = True` means the end stop was missing or unmatched, so we assumed the typical ride length of that route. Check how large that share is.

In [ ]:
pat.to_parquet(P / 'patterns.parquet'); hourly.to_parquet(P / 'route_hourly.parquet'); stops.to_parquet(P / 'stops.parquet'); sm.to_parquet(P / 'stop_map.parquet')
tx = tx[['route_n', 'route_raw', 'dir', 'b', 'a', 'pax', 'ts', 'date', 'hour', 'dow', 'is_ac', 'vehicle', 'imputed_end']]
tx.to_parquet(P / 'tickets_resolved.parquet'); print('saved')